# 2 — Quick-Start Tutorial

This notebook walks through the core `chemical_reservoir` workflow:

1. Install / import the package
2. Load a reaction network
3. Define a target ODE
4. Train the reservoir and evaluate
5. Visualise results
6. Try different networks and problems

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import chemical_reservoir as cr

print(f"chemical_reservoir v{cr.__version__}")

---
## 2.1  Loading a Reaction Network

Networks are defined as JSON files.  The package ships with several built-in
topologies under `chemical_reservoir/network_configs/`.

In [ ]:
# List built-in networks
from chemical_reservoir.networks import list_builtin_networks
print("Built-in networks:", list_builtin_networks())

In [ ]:
# Load a network from JSON and create a reservoir
reservoir, config = cr.io.load_reservoir_from_json(
    "../network_configs/cascade.json"
)

print(f"Network: {config['name']}")
print(f"Species: {config['all_species']}")
print(f"Reactions: {len(config['reactions'])}")
print(f"Input channels: {config['input_channels']}")
print(f"Output channels: {config['output_channels']}")

In [ ]:
# Visualise the reaction network topology
from chemical_reservoir import ReactionNetworkVisualizer

viz = ReactionNetworkVisualizer(reservoir)
fig = viz.plot_bipartite_network(title=f"{config['name']} reaction network")
plt.show()

---
## 2.2  Defining a Target ODE

The package provides factory functions for common ODE test problems.

In [ ]:
# Create the Van der Pol oscillator
problem = cr.problems.create_vanderpol_problem(mu=1.5)

print(f"Problem: {problem.name}")
print(f"Variables: {problem.variable_names}")
print(f"Time span: {problem.t_span}")
print(f"Initial conditions: {problem.y0}")

In [ ]:
# Solve and visualise the ground truth
t = np.linspace(*problem.t_span, 500)
y_true = problem.solve_numerical(t)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for i, name in enumerate(problem.variable_names):
    ax1.plot(t, y_true[:, i], label=name)
ax1.set(xlabel='t', ylabel='y', title=f'{problem.name} — Time Series')
ax1.legend()

ax2.plot(y_true[:, 0], y_true[:, 1], 'b-', alpha=0.7)
ax2.plot(y_true[0, 0], y_true[0, 1], 'go', ms=8, label='Start')
ax2.set(xlabel=problem.variable_names[0], ylabel=problem.variable_names[1],
        title='Phase Portrait')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 2.3  Training the Reservoir

Call `reservoir.solve_ode()` (open-loop) or `reservoir.solve_ode_closed_loop()` (closed-loop) to:
1. Solve the target ODE on the training interval
2. Drive the chemical reservoir with the ODE solution as input (teacher forcing)
3. Collect reservoir concentration features
4. Train a ridge-regression readout
5. Predict on the test interval — in **open-loop** mode, predictions use a single
   forward pass; in **closed-loop** mode, each prediction is fed back to modulate
   the input signal at the next step.

Below we use `solve_ode()` (open-loop) for simplicity.  See **Notebook 05** for
a side-by-side comparison of both modes.

In [ ]:
results = reservoir.solve_ode(
    problem,
    n_train=400,
    n_test=200,
    test_horizon_factor=0.5,  # test interval = 50% of training span
)

print("Result keys:", list(results.keys()))

---
## 2.4  Evaluating Performance

In [ ]:
metrics = cr.MetricsCalculator.calculate_all(
    y_true=results['y_true'],
    y_pred=results['y_pred'],
    t=results['t_all'],
    n_train=len(results['t_train']),
    variable_names=problem.variable_names,
)

print(f"{'Metric':<20} {'Train':>12} {'Test':>12}")
print('-' * 46)
print(f"{'RMSE':<20} {metrics.train_rmse:>12.4e} {metrics.test_rmse:>12.4e}")
print(f"{'R²':<20} {metrics.train_r2:>12.4f} {metrics.test_r2:>12.4f}")
print(f"{'Pearson r':<20} {metrics.train_pearson:>12.4f} {metrics.test_pearson:>12.4f}")

---
## 2.5  Visualisation

In [ ]:
# Quick manual plot
n_train = len(results['t_train'])
t_all = results['t_all']

fig, axes = plt.subplots(len(problem.variable_names), 1,
                         figsize=(12, 3 * len(problem.variable_names)),
                         sharex=True)
if len(problem.variable_names) == 1:
    axes = [axes]

for i, name in enumerate(problem.variable_names):
    ax = axes[i]
    ax.plot(t_all, results['y_true'][:, i], 'b-', lw=2, label='True', alpha=0.7)
    ax.plot(t_all, results['y_pred'][:, i], 'r--', lw=2, label='Predicted', alpha=0.7)
    ax.axvline(t_all[n_train - 1], color='k', ls=':', lw=1, label='Train/Test')
    ax.set_ylabel(name, fontsize=12)
    ax.legend(loc='upper right')

axes[-1].set_xlabel('t', fontsize=12)
fig.suptitle(f'{problem.name}: Reservoir Prediction', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
n_train = len(results['t_train'])
t_all = results['t_all']
y_true = results['y_true']
y_pred = results['y_pred']
var_names = problem.variable_names
errors = np.sqrt(np.sum((y_true - y_pred)**2, axis=1))

# --- 1. Error over time ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(t_all, errors + 1e-10, 'b-', lw=2)
ax.axvline(results['t_train'][-1], color='red', ls=':', lw=2)
ax.fill_between(t_all[:n_train], 1e-10, errors[:n_train], alpha=0.3, color='blue', label='Training')
ax.fill_between(t_all[n_train:], 1e-10, errors[n_train:], alpha=0.3, color='orange', label='Testing')
ax.set(xlabel='Time', ylabel='RMSE', title='Error Evolution')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. Error metrics (RMSE, MAE, NRMSE) ---
fig, ax = plt.subplots(figsize=(6, 4))
metric_names = ['RMSE', 'MAE', 'NRMSE']
train_vals = [metrics.train_rmse, metrics.train_mae, metrics.train_nrmse]
test_vals = [metrics.test_rmse, metrics.test_mae, metrics.test_nrmse]
x = np.arange(len(metric_names))
width = 0.35
ax.bar(x - width/2, train_vals, width, label='Train', color='steelblue')
ax.bar(x + width/2, test_vals, width, label='Test', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set(ylabel='Value', title='Error Metrics')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# --- 3. Correlation metrics (R² and Pearson) ---
fig, ax = plt.subplots(figsize=(5, 4))
corr_names = ['R²', 'Pearson']
train_corr = [metrics.train_r2, metrics.train_pearson]
test_corr = [metrics.test_r2, metrics.test_pearson]
width = 0.35
ax.bar(np.arange(2) - width/2, train_corr, width, label='Train', color='steelblue')
ax.bar(np.arange(2) + width/2, test_corr, width, label='Test', color='darkorange')
ax.set_xticks(np.arange(2))
ax.set_xticklabels(corr_names)
ax.set(ylabel='Value', title='Correlation Metrics', ylim=[0, 1.1])
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# --- 4. Per-variable RMSE ---
if metrics.per_variable_train_rmse:
    fig, ax = plt.subplots(figsize=(6, 4))
    vars_list = list(metrics.per_variable_train_rmse.keys())
    train_rmse = [metrics.per_variable_train_rmse[v] for v in vars_list]
    test_rmse = [metrics.per_variable_test_rmse.get(v, 0) for v in vars_list]
    x = np.arange(len(vars_list))
    width = 0.35
    ax.bar(x - width/2, train_rmse, width, label='Train', color='steelblue')
    ax.bar(x + width/2, test_rmse, width, label='Test', color='darkorange')
    ax.set_xticks(x)
    ax.set_xticklabels(vars_list)
    ax.set(ylabel='RMSE', title='Per-Variable RMSE')
    ax.legend()
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

In [ ]:
# --- 5. True vs Predicted scatter ---
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_true[:n_train].flatten(), y_pred[:n_train].flatten(),
           alpha=0.3, s=10, c='blue', label='Train')
ax.scatter(y_true[n_train:].flatten(), y_pred[n_train:].flatten(),
           alpha=0.3, s=10, c='orange', label='Test')
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
ax.plot(lims, lims, 'k--', lw=2, label='Ideal')
ax.set(xlabel='True', ylabel='Predicted', title='Prediction Accuracy')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6. Phase space: true vs predicted ---
if len(var_names) >= 2:
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(y_true[:, 0], y_true[:, 1], 'b-', lw=2, alpha=0.7, label='True')
    ax.plot(y_pred[:, 0], y_pred[:, 1], 'r--', lw=2, alpha=0.7, label='Predicted')
    ax.set(xlabel=var_names[0], ylabel=var_names[1], title='Phase Space')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- 7. Error distribution ---
fig, ax = plt.subplots(figsize=(7, 4))
all_errors = (y_true - y_pred).flatten()
ax.hist(all_errors, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
ax.axvline(0, color='red', ls='--', lw=2)
ax.set(xlabel='Prediction Error', ylabel='Density', title='Error Distribution')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2.6  Trying Different Problems

Swap the problem factory to try other ODEs.

In [ ]:
# Brusselator
brusselator = cr.problems.create_brusselator_problem()
res_bruss = reservoir.solve_ode(brusselator, n_train=400, n_test=200)

met_bruss = cr.MetricsCalculator.calculate_all(
    y_true=res_bruss['y_true'],
    y_pred=res_bruss['y_pred'],
    t=res_bruss['t_all'],
    n_train=len(res_bruss['t_train']),
    variable_names=brusselator.variable_names,
)
print(f"Brusselator — Test RMSE: {met_bruss.test_rmse:.4e},  R²: {met_bruss.test_r2:.4f}")

In [ ]:
# Lotka-Volterra
lv = cr.problems.create_lotka_volterra_problem()
res_lv = reservoir.solve_ode(lv, n_train=400, n_test=200)

met_lv = cr.MetricsCalculator.calculate_all(
    y_true=res_lv['y_true'],
    y_pred=res_lv['y_pred'],
    t=res_lv['t_all'],
    n_train=len(res_lv['t_train']),
    variable_names=lv.variable_names,
)
print(f"Lotka-Volterra — Test RMSE: {met_lv.test_rmse:.4e},  R²: {met_lv.test_r2:.4f}")

---
## 2.7  Trying Different Networks

Compare how network topology affects performance.

In [ ]:
from chemical_reservoir.networks import list_builtin_networks
from chemical_reservoir.io import load_reservoir_from_json
from pathlib import Path

network_dir = Path("../network_configs")
problem = cr.problems.create_vanderpol_problem()

print(f"{'Network':<25} {'Test RMSE':>12} {'Test R²':>10}")
print('-' * 50)

for name in ['minimal', 'simple_autocatalytic', 'cascade', 'dense']:
    json_path = network_dir / f"{name}.json"
    if not json_path.exists():
        continue
    res, cfg = load_reservoir_from_json(json_path)
    results = res.solve_ode(problem, n_train=400, n_test=200)
    met = cr.MetricsCalculator.calculate_all(
        y_true=results['y_true'], y_pred=results['y_pred'],
        t=results['t_all'], n_train=len(results['t_train']),
        variable_names=problem.variable_names,
    )
    print(f"{name:<25} {met.test_rmse:>12.4e} {met.test_r2:>10.4f}")

---
## Next Steps

- **Notebook 03**: PDE solving, Bayesian optimisation, sensitivity analysis,
  custom network design.
- **Notebook 04**: API reference with detailed class documentation.
- **Notebook 05**: Open-loop vs closed-loop prediction comparison.